In [ ]:
import xarray as xr
import ndsl.dsl.gt4py_utils as gt_utils
from ndsl import GridSizer, Quantity, QuantityFactory, TileCommunicator, TilePartitioner, NullComm, SubtileGridSizer
import ndsl.constants as constants

from pyshield.radiation import RadiationState, RadiationConfig, RTE_RRTMGPDriver
from pyshield.physics_state import SurfaceState
from pyshield._config import PHYSICS_PACKAGES, PhysicsConfig
import numpy as np
import matplotlib.pyplot as plt
import datetime

In [ ]:
ds = xr.open_dataset("RESTART/restart_physics_state_0.nc")
ds2 = xr.open_dataset("RESTART/restart_dycore_state_0.nc")

In [ ]:
schemes = PHYSICS_PACKAGES
nx_tile=20
ny_tile=20
nz=79
n_halo=3
date = datetime.datetime(2020, 1, 1, 12, tzinfo=datetime.timezone.utc)

In [ ]:
rank = 0

comm = NullComm(rank, 1)
communicator = TileCommunicator.from_layout(comm=comm, layout=(1,1))

sizer = SubtileGridSizer.from_tile_params(
    nx_tile=nx_tile,
    ny_tile=ny_tile,
    nz=nz,
    n_halo=n_halo,
    extra_dim_lengths={},
    layout=(1,1),
    tile_partitioner=communicator.partitioner.tile,
    tile_rank=communicator.tile.rank,
)
quantity_factory = QuantityFactory.from_backend(
    sizer, backend="numpy"
)

In [ ]:
from ndsl import StencilFactory, CompilationConfig, StencilConfig, GridIndexing

In [ ]:
comconf = CompilationConfig()
comconf.validate_args = False
sconf = StencilConfig(compilation_config=comconf)
grid_indexing = GridIndexing.from_sizer_and_communicator(sizer=sizer, comm=communicator)
stencil_factory = StencilFactory(config=sconf, grid_indexing=grid_indexing, comm=comm)

In [ ]:
from ndsl.grid import (
    AngleGridData,
    ContravariantGridData,
    DampingCoefficients,
    DriverGridData,
    GridData,
    HorizontalGridData,
    MetricTerms,
    VerticalGridData,
)
from pathlib import Path

In [ ]:
metric_terms = MetricTerms(
    quantity_factory=quantity_factory,
    communicator=communicator,
    grid_type=4,
    eta_file="eta79.nc",
)
horizontal_data = HorizontalGridData.new_from_metric_terms(metric_terms)
vertical_data = VerticalGridData.new_from_metric_terms(metric_terms)
contravariant_data = ContravariantGridData.new_from_metric_terms(metric_terms)
angle_data = AngleGridData.new_from_metric_terms(metric_terms)
grid_data = GridData(
    horizontal_data=horizontal_data,
    vertical_data=vertical_data,
    contravariant_data=contravariant_data,
    angle_data=angle_data,
)

In [ ]:
grid_data.lon_agrid.field[:] = grid_data.lon.field[:-1,:-1]
grid_data.lat_agrid.field[:] = grid_data.lat.field[:-1,:-1]

In [ ]:
conf = PhysicsConfig
radconf = RadiationConfig(
    deltsw = 3600.0,
    delt_rad = 3600.0,
    date=date,
    fhswr=1.0,
    fhlwr=1.0,
    isolar=10,
    icmphys=4,
    ico2flg=0,
    ioznflg=1,
    ictmflg=-1,
    ialbflg=-1,
    iemsflg=0,
    ldisable_radiation_quasi_sea_ice=False,
    solar_constant_file=Path("global_solarconstant_noaa_an.txt"),
    input_dir=Path("../../test_data/"),
    aerosol_file=Path("../../test_data/"),
    sollat=0.0,
    nstp=6,
    ivflip=1,
    lcnorm=False,
    lcrick=False,
    gfs_cloud_overlap=False,
)

In [ ]:
from pyshield.stencils.physics import calc_sigma

In [ ]:
sigma = calc_sigma(grid_data.ak.data, grid_data.bk.data, 0)
gridlon = grid_data.lon_agrid
gridlat = grid_data.lat_agrid

In [ ]:
rad = RTE_RRTMGPDriver(config=radconf, gridlon=gridlon, gridlat=gridlat, sigma=sigma[::-1], quantity_factory=quantity_factory, stencil_factory=stencil_factory)

In [ ]:
state = RadiationState.init_zeros(quantity_factory, np)
sstate = SurfaceState.init_zeros(quantity_factory)
type(state.prsi)

In [ ]:
prsi = ds.prsi.values[3:-4,3:-4,::-1] # level pressure
prsl = (prsi[:,:,1:] - prsi[:,:,:-1]) / np.log(prsi[:,:,1:] / prsi[:,:,:-1])
pt = ds.pt.values[3:-4,3:-4,-2::-1] # Layer temperature
delp = ds.delp.values[3:-4,3:-4,-2::-1]
delz = -1.* ds.delz.values[3:-4,3:-4,-2::-1]
dz = -1.* ds.dz.values[3:-4,3:-4,-2::-1]
qvap = ds.qvapor.values[3:-4,3:-4,-2::-1]
qliq = ds.qliquid.values[3:-4,3:-4,-2::-1]
qice = ds.qice.values[3:-4,3:-4,-2::-1]
qcld = ds.qcld.values[3:-4,3:-4,-2::-1]
phii = ds.phii.values[3:-4,3:-4,::-1]
ptlev = np.zeros_like(prsi)
ptlev[:,:,1:-1] = pt[:,:,:-1] + (pt[:,:,1:] - pt[:,:,:-1]) * (np.log(prsi[:,:,1:-1]) - np.log(prsl[:,:,:-1])) / (np.log(prsl[:,:,1:]) - np.log(prsl[:,:,:-1]))
ptlev[:,:,-1] = pt[:,:,-1]
ptlev[:,:,0] = pt[:,:,0]
qo3mr = ds.qo3mr.values[3:-4,3:-4,-2::-1]
tskin = ptlev[:,:,-1]

In [ ]:
state.prsi.view[:] = prsi
state.prsl.view[:] = prsl
state.tlyr.view[:] = pt
state.tlvl.view[:] = ptlev
state.tsfc.view[:] = tskin
state.qvapor.view[:] = qvap
state.qliquid.view[:] = qliq
state.qice.view[:] = qice
state.qcld.view[:] = qcld
state.qo3mr.view[:] = qo3mr

In [ ]:
rad.step_radiation(state, sstate, date)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
levels = np.arange(nz+1)

In [ ]:
plt.plot(state.fswd.view[0,0,:], levels, label="SW all-sky flux Down")
plt.plot(state.fswu.view[0,0,:], levels, label="SW all-sky flux Up")
plt.plot(state.flwd.view[0,0,:], levels, label="LW all-sky flux Down")
plt.plot(state.flwu.view[0,0,:], levels, label="LW all-sky flux Up")
plt.legend(frameon=False)

In [ ]:
newdate = date + datetime.timedelta(seconds=radconf.delt_rad)

In [ ]:
%%time
rad.step_radiation(state, sstate, newdate)

In [ ]:
plt.plot(state.fswd.view[0,0,:], levels, label="SW all-sky flux Down")
plt.plot(state.fswu.view[0,0,:], levels, label="SW all-sky flux Up")
plt.plot(state.flwd.view[0,0,:], levels, label="LW all-sky flux Down")
plt.plot(state.flwu.view[0,0,:], levels, label="LW all-sky flux Up")
plt.legend(frameon=False)

In [ ]:
print(newdate)

In [ ]:
%%time
for i in range(10):
    newdate = newdate + datetime.timedelta(seconds=radconf.delt_rad)
    print(newdate)
    rad.step_radiation(state, sstate, newdate)

In [ ]:
plt.plot(state.fswd.view[0,0,:], levels, label="SW all-sky flux Down")
plt.plot(state.fswu.view[0,0,:], levels, label="SW all-sky flux Up")
plt.plot(state.flwd.view[0,0,:], levels, label="LW all-sky flux Down")
plt.plot(state.flwu.view[0,0,:], levels, label="LW all-sky flux Up")
plt.legend(frameon=False)

In [ ]:
state.mu0.field[:]